This notebook runs the full multi-asset experiment suite (Experiments A through E) across 10 bank stocks — JPM, C, WFC, GS, MS, PNC, USB, FITB, MTB, and BAC. It imports and calls the existing pipeline from `run_multiasset_experiments.py` and `scripts/`, which must be on the Python path. All trained weights are cached under `results_multiasset/_cache/`; re-running any cell skips training for experiments that have already completed. Final results, NLL tables, and a markdown summary are written to `results_multiasset/`.

In [ ]:
import sys
import torch
import numpy as np

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"

print(f"Device           : {device_name}")
print(f"torch version    : {torch.__version__}")
print(f"numpy version    : {np.__version__}")
print(f"Python version   : {sys.version.split()[0]}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import sys
from pathlib import Path

project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.config import make_config
from run_multiasset_experiments import run_all_experiments, RESULTS_DIR, TARGET_STOCKS

cfg = make_config()
print(f"horizons  : {cfg.horizons}")
print(f"seeds     : {cfg.seeds}")
print(f"swa_epochs: {cfg.swa_epochs}")
print(f"max_epochs: {cfg.max_epochs}  patience: {cfg.patience}")

In [ ]:
import contextlib
import sys
from pathlib import Path

from run_multiasset_experiments_ext import run_all_experiments_ext

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
log_path = RESULTS_DIR / "run.log"

class _Tee:
    def __init__(self, *streams):
        self._streams = streams
    def write(self, data):
        for s in self._streams:
            s.write(data)
            s.flush()
    def flush(self):
        for s in self._streams:
            s.flush()

with open(log_path, "a") as _log_file:
    with contextlib.redirect_stdout(_Tee(sys.stdout, _log_file)):
        run_all_experiments(cfg)
        run_all_experiments_ext(cfg)

print(f"\nLog written to {log_path}")

In [ ]:
from pathlib import Path

RESULTS_DIR_PATH = Path("results_multiasset")

summary_path = RESULTS_DIR_PATH / "summary.md"
assert summary_path.exists(), f"summary.md not found at {summary_path}"
print(f"summary.md found: {summary_path}")

missing = []
for ticker in TARGET_STOCKS:
    for subdir in ["baseline", "swa_bestsigma"]:
        p = RESULTS_DIR_PATH / ticker / subdir
        if not p.exists():
            missing.append(str(p))

if missing:
    print("MISSING output directories:")
    for m in missing:
        print(f"  {m}")
else:
    print("All 20 expected ticker/experiment directories present.")

weight_files = list(RESULTS_DIR_PATH.rglob("weights/*.pt"))
print(f"Total .pt weight files saved: {len(weight_files)}")

failed_markers = list(RESULTS_DIR_PATH.rglob("FAILED"))
if failed_markers:
    print(f"\nWARNING: {len(failed_markers)} FAILED marker(s) found:")
    for f in failed_markers:
        print(f"  {f}")
else:
    print("No FAILED markers — all experiments completed successfully.")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(summary_path.read_text())

In [ ]:
"""Cell 5 — Extended experiment completion manifest.

Prints expected vs completed cell counts for each extended experiment.
A cell is 'complete' if nll_metrics.csv has at least one row for that
(ticker, model, h, N) combination.
"""
import pandas as pd
from pathlib import Path

RESULTS_DIR_PATH = Path("results_multiasset")
TICKERS_EXT      = ["JPM", "C", "WFC", "GS", "MS", "PNC", "USB", "FITB", "MTB", "BAC"]
HORIZONS_EXT     = [1, 5, 10, 21, 63]
BINS_EXT         = [4, 10, 20, 35, 55]
REGIMES_EXT      = ["full", "macro_only", "own_only", "other_banks_only"]
K_LIST           = [1, 2, 3, 5, 10]
D_HORIZONS       = [1, 5, 21, 63]
E_HORIZONS       = [1, 21]
SEEDS            = [42, 7, 123]

def _count_complete(nll_csv: Path, group_cols):
    """Count unique completed (group_cols) tuples in nll_metrics.csv."""
    if not nll_csv.exists():
        return 0, set()
    try:
        df = pd.read_csv(nll_csv).dropna(subset=["nll_test"])
        completed = set(df[group_cols].drop_duplicates().itertuples(index=False, name=None))
        return len(completed), completed
    except Exception:
        return 0, set()

print("=" * 65)
print("EXTENDED EXPERIMENT COMPLETION MANIFEST")
print("=" * 65)

# A-EXT: 10 tickers × 5h × 5N × 2 models = 500 unique cells
print("\n--- A-EXT (baseline_ext) ---")
expected_Aext = set()
for t in TICKERS_EXT:
    for h in HORIZONS_EXT:
        for N in BINS_EXT:
            for m in ["state_cond", "state_free"]:
                expected_Aext.add((t, m, h, N))
completed_Aext = set()
for t in TICKERS_EXT:
    p = RESULTS_DIR_PATH / t / "baseline_ext" / "nll_metrics.csv"
    _, done = _count_complete(p, ["model", "h", "N"])
    completed_Aext |= {(t,) + cell for cell in done}
print(f"  Completed: {len(completed_Aext)}/{len(expected_Aext)}", end="")
missing_A = expected_Aext - completed_Aext
print(f"  ({len(missing_A)} incomplete)" if missing_A else "  ✓")

# C-EXT: same structure as A-EXT
print("\n--- C-EXT (swa_bestsigma_ext) ---")
completed_Cext = set()
for t in TICKERS_EXT:
    p = RESULTS_DIR_PATH / t / "swa_bestsigma_ext" / "nll_metrics.csv"
    _, done = _count_complete(p, ["model", "h", "N"])
    completed_Cext |= {(t,) + cell for cell in done}
print(f"  Completed: {len(completed_Cext)}/{len(expected_Aext)}", end="")
missing_C = expected_Aext - completed_Cext
print(f"  ({len(missing_C)} incomplete)" if missing_C else "  ✓")

# D-EXT: 5k × 4h = 20 unique cells (JPM)
print("\n--- D-EXT (higher_order_ext) ---")
expected_Dext = {(f"ho_k{k}", h, 55) for k in K_LIST for h in D_HORIZONS}
p_d = RESULTS_DIR_PATH / "JPM" / "higher_order_ext" / "nll_metrics.csv"
n_d, done_d = _count_complete(p_d, ["model", "h", "N"])
print(f"  Completed: {n_d}/{len(expected_Dext)}", end="")
missing_D = expected_Dext - done_d
print(f"  ({len(missing_D)} incomplete)" if missing_D else "  ✓")

# E-EXT: 10 tickers × 4 regimes × 2h = 80 unique cells
print("\n--- E-EXT (conditioning_regime_ext) ---")
expected_Eext = set()
for t in TICKERS_EXT:
    for r in REGIMES_EXT:
        for h in E_HORIZONS:
            expected_Eext.add((t, f"state_cond_{r}", h, 55))
completed_Eext = set()
for t in TICKERS_EXT:
    p = RESULTS_DIR_PATH / t / "conditioning_regime_ext" / "nll_metrics.csv"
    _, done = _count_complete(p, ["model", "h", "N"])
    completed_Eext |= {(t,) + cell for cell in done}
print(f"  Completed: {len(completed_Eext)}/{len(expected_Eext)}", end="")
missing_E = expected_Eext - completed_Eext
print(f"  ({len(missing_E)} incomplete)" if missing_E else "  ✓")

# F-NEW: cross_asset_diagnostics.csv
print("\n--- F-NEW (cross_asset_diagnostics) ---")
cross_p = RESULTS_DIR_PATH / "cross_asset_diagnostics.csv"
if cross_p.exists():
    df_cross = pd.read_csv(cross_p)
    tickers_found = sorted(df_cross["ticker"].unique()) if "ticker" in df_cross.columns else []
    print(f"  Tickers in CSV: {tickers_found}")
    print(f"  Rows: {len(df_cross)}")
else:
    print("  WARNING: cross_asset_diagnostics.csv not found")

# Operator TS file counts
print("\n--- Operator diagnostics time series ---")
for exp_tag in ["baseline_ext", "swa_bestsigma_ext"]:
    ts_files = list(RESULTS_DIR_PATH.glob(f"*/{ exp_tag}/operator_ts_*.csv"))
    print(f"  {exp_tag}: {len(ts_files)} operator_ts CSVs")
    snap_files = list(RESULTS_DIR_PATH.glob(f"*/{exp_tag}/snapshots/*.npy"))
    print(f"  {exp_tag}: {len(snap_files)} snapshot .npy files")

# FAILED markers in ext experiments
failed_ext = list(RESULTS_DIR_PATH.glob("*/baseline_ext/FAILED")) + \
             list(RESULTS_DIR_PATH.glob("*/swa_bestsigma_ext/FAILED")) + \
             list(RESULTS_DIR_PATH.glob("*/higher_order_ext/FAILED")) + \
             list(RESULTS_DIR_PATH.glob("*/conditioning_regime_ext/FAILED"))
if failed_ext:
    print(f"\nWARNING: {len(failed_ext)} FAILED marker(s) in ext experiments:")
    for f in failed_ext:
        print(f"  {f}")
else:
    print("\nNo FAILED markers in ext experiments.")

print("\n" + "=" * 65)
total_complete = len(completed_Aext) + len(completed_Cext) + n_d + len(completed_Eext)
total_expected = len(expected_Aext) + len(expected_Aext) + len(expected_Dext) + len(expected_Eext)
print(f"TOTAL: {total_complete}/{total_expected} cells complete across A-EXT, C-EXT, D-EXT, E-EXT")
